In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
df_train = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
df_test = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

In [ ]:
df_train.shape

In [ ]:
df_train.info()
df_test.info()

In [ ]:
df_train.drop(columns=["model_a", "model_b"], inplace=True)

In [ ]:
def random_split(df_train, train_frac, validation_frac):
    df = df_train.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:]

    return train_df, validation_df

train_df, validation_df = random_split(df_train, 0.7, 0.3)

In [ ]:
print(len(train_df), len(validation_df))

In [ ]:
df_train.info()

In [ ]:
import tiktoken
import matplotlib.pyplot as plt

tokenizer = tiktoken.get_encoding("gpt2")
text_columns = ["prompt", "response_a", "response_b"]

# Combine text columns into one string per row
text_data = df_train[text_columns].astype(str).agg(" ".join, axis=1)

# Tokenize each combined text
encoded_texts = [
    tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    for text in text_data
]

# Calculate max token length
max_length = max(len(enc) for enc in encoded_texts)
print("Max token length:", max_length)

# Plot distribution of token lengths
token_lengths = [len(enc) for enc in encoded_texts]
plt.hist(token_lengths, bins=100)
plt.title("Distribution of Token Lengths")
plt.xlabel("Token Length")
plt.ylabel("Frequency")
plt.show()


In [ ]:
num_truncated = sum(len(enc) > 1024 for enc in encoded_texts)
lengths = torch.tensor([len(enc) for enc in encoded_texts], dtype=torch.float)
median_len = torch.quantile(lengths, 0.75)
print(f"{median_len} is the median length of the encoded texts ")
print(f"{num_truncated} samples will be truncated.")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset

class LLMClassificationDataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer, max_length=960, pad_token_id=50256):
        self.data = data.copy()  # DataFrame passed directly
        self.max_length = max_length
        self.text_columns = ["prompt", "response_a", "response_b"]
        
        # Combine text columns into one string per row
        self.text_data = self.data[self.text_columns].astype(str).agg(" ".join, axis=1)
        
        # Tokenize each combined text
        self.encoded_texts = [
        tokenizer.encode(text, allowed_special={"<|endoftext|>"})[:max_length]
        for text in self.text_data
        ]

        
        # Pad sequences to max_length
        self.encoded_texts = [
            tokens + [pad_token_id] * (max_length - len(tokens))
            for tokens in self.encoded_texts
        ]
        
        # Extract labels as floats (for probabilities)
        self.labels = self.data[["winner_model_a", "winner_model_b", "winner_tie"]].values.astype(float)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        input_ids = torch.tensor(self.encoded_texts[idx], dtype=torch.long)
        labels = torch.tensor(self.labels[idx], dtype=torch.float)
        return input_ids, labels


In [ ]:
class LLMClassificationTestDataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer, max_length=960, pad_token_id=50256):
        self.data = data.copy()  # DataFrame passed directly
        self.max_length = max_length
        self.text_columns = ["prompt", "response_a", "response_b"]
        
        # Combine text columns into one string per row
        self.text_data = self.data[self.text_columns].astype(str).agg(" ".join, axis=1)
        
        # Tokenize each combined text
        self.encoded_texts = [
        tokenizer.encode(text, allowed_special={"<|endoftext|>"})[:max_length]
        for text in self.text_data
        ]
        
        # Pad sequences to max_length
        self.encoded_texts = [
            tokens + [pad_token_id] * (max_length - len(tokens))
            for tokens in self.encoded_texts
        ]
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        input_ids = torch.tensor(self.encoded_texts[idx], dtype=torch.long)
        return input_ids


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

dataset_train = LLMClassificationDataset(train_df, tokenizer, max_length=1024, pad_token_id=50256)
dataset_val = LLMClassificationDataset(validation_df, tokenizer, max_length=1024, pad_token_id=50256)
dataset_test = LLMClassificationTestDataset(df_test, tokenizer, max_length=1024, pad_token_id=50256)
input_ids, labels = dataset_train[2]

print(input_ids.shape)  # torch.Size([max_length])
print(labels)           # tensor([1.0, 0.0, 0.0]) for example


In [ ]:
from torch.utils.data import DataLoader

batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=dataset_train,
    batch_size=8,             
    shuffle=True,
    num_workers=0,             
    drop_last=True
)

val_loader = DataLoader(
    dataset=dataset_val,
    batch_size = 8,
    num_workers=0,
    drop_last=True
    
)

test_loader = DataLoader(
    dataset=dataset_test,
    batch_size = 1,
    num_workers=0,
    drop_last=False
    
)

In [ ]:
# GPT2 architecture


cfg = {
    "vocab_size":50257,
    "context_length":1024,
    "emb_dim":768,
    "n_layer":12,
    "n_head":12,
    "dropout":0.1,
    "qkv_bias":True
}

assert dataset_train.max_length <= cfg["context_length"]

import torch.nn as nn
class MaskedMultiheadAttention(nn.Module):
    
    def __init__(self, din, dout, context_length, num_heads, dropout_rate, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.dout = dout
        self.head_dim  = dout // num_heads
        self.Wq = torch.nn.Linear(din, dout, bias=qkv_bias) # dout is the user defiend but din should strictly be equal to dim = embed_size for matrix mul
        self.Wk = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.Wv = torch.nn.Linear(din, dout, bias=qkv_bias)
        self.out_proj = torch.nn.Linear(dout, dout)
        self.dropout = torch.nn.Dropout(dropout_rate)
        
        mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
        self.register_buffer("mask", mask)
        
    def forward(self, x):
        batch, n_tokens, din = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        Q = Q.view(batch, n_tokens, self.num_heads, self.head_dim)
        V = V.view(batch, n_tokens, self.num_heads, self.head_dim)
        K = K.view(batch, n_tokens, self.num_heads, self.head_dim)
        
        Q = Q.transpose(1,2)
        V = V.transpose(1,2)
        K = K.transpose(1,2)
        
        attention_scores = Q @ K.transpose(2,3)
        masked = attention_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        atten_weights = torch.softmax(masked/self.head_dim ** 0.5, dim=-1)
        atten_weights = self.dropout(atten_weights)
        context_vector = (atten_weights @ V).transpose(1,2)
        context_vector = context_vector.contiguous().view(batch, n_tokens, self.dout)
        context_vector =  self.out_proj(context_vector)
        
        return context_vector




class LayerNorm(nn.Module):
    
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(embed_dim))
        self.shift = nn.Parameter(torch.zeros(embed_dim))
        self.espilon = 1e-7
        
    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True, unbiased=True)
        norm_x = (x - mean)/torch.sqrt(var + self.espilon )
        return self.scale * norm_x + self.shift
    
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x ** 3)))

    
class FeedForward(nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4*cfg['emb_dim']),
            GELU(),
            nn.Linear(4*cfg['emb_dim'], cfg['emb_dim'])
        )
        
    def forward(self, x):
        return self.layer(x)




class Transformer(nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        self.attention = MaskedMultiheadAttention(
            din = cfg["emb_dim"], 
            dout = cfg["emb_dim"], 
            context_length = cfg["context_length"], 
            num_heads = cfg["n_head"], 
            dropout_rate = cfg["dropout"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(embed_dim = cfg["emb_dim"])
        self.norm2 = LayerNorm(embed_dim = cfg["emb_dim"])      
        self.drop = nn.Dropout(cfg["dropout"])
        
    def forward(self, x):
        shortcut = x
        x = self.norm1(x) # pre normalization results in better reults than post normalization
        x = self.attention(x)
        x = self.drop(x)
        x = x + shortcut
        
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop(x)
        x = x + shortcut
        
        return x





class GPT(nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        self.embed_layer = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_layer = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["dropout"])
        
        self.tf_blocks = nn.Sequential(*[Transformer(cfg) for _ in range(cfg["n_layer"])])
        
        self.norm = LayerNorm(cfg["emb_dim"])
        
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self, in_ids):
        batch, seq_len = in_ids.shape
        token_embed = self.embed_layer(in_ids)
        pos_ids = torch.arange(seq_len, device=in_ids.device)  # fixes the device mismatch
        pos_encoding = self.pos_layer(pos_ids)
        x = token_embed + pos_encoding
        x = self.dropout(x)
        x = self.tf_blocks(x)
        x = self.norm(x)
        return self.out_head(x)  # shape: (batch_size, num_classes)
  


In [ ]:
def crossentropyloss(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)

    logits = model(input_batch)[:,-1,:]
    loss = nn.functional.cross_entropy(logits, target_batch)
    return loss

def calc_loss(dataloader, model, devoce, num_batches=None):
    total_loss = 0
    if len(dataloader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(dataloader)
    else :
        num_batches = min(num_batches, len(dataloader))
    for i, (input_batch, target_batch) in enumerate(dataloader):
        if i < num_batches:
            loss = crossentropyloss(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else :
            break
    return  total_loss / num_batches

In [ ]:
def calc_acc(dataloader, model, device, num_batches=None):
    model.eval()
    correct_pred, total_labels = 0, 0

    if num_batches is None:
        num_batches = len(dataloader)
    else:
        num_batches = min(num_batches, len(dataloader))

    for i, (input_batch, target_batch) in enumerate(dataloader):
        if i >= num_batches:
            break

        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device).float()

        with torch.no_grad():
            logits = model(input_batch)[:, -1, :]  # [batch_size, 3]
            probs = torch.sigmoid(logits)          # [batch_size, 3]
            predicted_labels = (probs > 0.5).float()

        correct_pred += (predicted_labels == target_batch).sum().item()
        total_labels += target_batch.numel()

    return correct_pred / total_labels  # element-wise accuracy


In [ ]:
import os
import json
import requests
import tensorflow as tf
from tqdm import tqdm

def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path)

    ## We have reached here until now ---> we have downloaded the files on our local machine.

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json")))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params

def download_file(url, destination):
    try:
        # Send a GET request to download the file, disabling SSL verification
        response = requests.get(url, stream=True, verify=False)

        # Get the total file size from headers, defaulting to 0 if not present
        file_size = int(response.headers.get("content-length", 0))

        # Check if file exists and has the same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return

        # Define the block size for reading the file
        block_size = 1024  # 1 Kilobyte

        # Initialize the progress bar with total file size
        progress_bar_description = url.split("/")[-1]  # Extract filename from URL
        with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
            # Open the destination file in binary write mode
            with open(destination, "wb") as file:
                # Iterate over the file data in chunks
                for chunk in response.iter_content(block_size):
                    progress_bar.update(len(chunk))  # Update progress bar
                    file.write(chunk)  # Write the chunk to the file

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        print(f"Please check the URL: {url}")

def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params


In [ ]:


settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
print(settings)
print(params.keys())

In [ ]:
# check wheter two tensors ro arrays(left and right) have same dim 
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch: left{left.shape}, right{right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_layer.weight = assign(gpt.pos_layer.weight, params['wpe'])
    gpt.embed_layer.weight = assign(gpt.embed_layer.weight, params['wte'])

    for b in range(len(params["blocks"])):
# weights of the multiheadattention block
        q_w, k_w, v_w = np.split(params["blocks"][b]["attn"]["c_attn"]["w"],3,axis=-1)
        gpt.tf_blocks[b].attention.Wq.weight = assign(gpt.tf_blocks[b].attention.Wq.weight, q_w.T)
        gpt.tf_blocks[b].attention.Wk.weight = assign(gpt.tf_blocks[b].attention.Wk.weight, k_w.T)
        gpt.tf_blocks[b].attention.Wv.weight = assign(gpt.tf_blocks[b].attention.Wv.weight, v_w.T)
# biases of the multiheadattention block
        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"],3,axis=-1)
        gpt.tf_blocks[b].attention.Wq.bias = assign(gpt.tf_blocks[b].attention.Wq.bias, q_b)
        gpt.tf_blocks[b].attention.Wk.bias = assign(gpt.tf_blocks[b].attention.Wk.bias, k_b)
        gpt.tf_blocks[b].attention.Wv.bias = assign(gpt.tf_blocks[b].attention.Wv.bias, v_b)
# weigthts and biases of the feedforward neural network
        gpt.tf_blocks[b].ff.layer[0].weight = assign(gpt.tf_blocks[b].ff.layer[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.tf_blocks[b].ff.layer[0].bias = assign(gpt.tf_blocks[b].ff.layer[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.tf_blocks[b].ff.layer[2].weight = assign(gpt.tf_blocks[b].ff.layer[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.tf_blocks[b].ff.layer[2].bias = assign(gpt.tf_blocks[b].ff.layer[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])
# weight and biases of the linear layer of the transformer 
        gpt.tf_blocks[b].attention.out_proj.weight = assign(gpt.tf_blocks[b].attention.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.tf_blocks[b].attention.out_proj.bias = assign(gpt.tf_blocks[b].attention.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])
# linear normalization 1 and 2 (scale and shift parameters)
        gpt.tf_blocks[b].norm1.scale = assign(gpt.tf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.tf_blocks[b].norm1.shift = assign(gpt.tf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.tf_blocks[b].norm2.scale = assign(gpt.tf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.tf_blocks[b].norm2.shift = assign(gpt.tf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])
# final normalization scale and shift parameters
    gpt.norm.scale = assign(gpt.norm.scale, params["g"])
    gpt.norm.shift = assign(gpt.norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
gpt = GPT(cfg)
gpt.eval();
load_weights_into_gpt(gpt, params)


In [ ]:
# Freeze the weights of gpt 
for param in gpt.parameters():
    param.requires_grad = False

# Replace the output layer with a linear layer
num_classes = 3
gpt.out_head = torch.nn.Linear(in_features=cfg["emb_dim"], out_features=num_classes)

for param in gpt.tf_blocks[-1].parameters():
    param.requires_grad = True

for param in gpt.norm.parameters():
    param.requires_grad = True

In [ ]:
# Count trainable vs frozen parameters
trainable_params = 0
frozen_params = 0
total_params = 0

print("=== Parameter Status ===")
for name, param in gpt.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"✅ TRAINABLE: {name} - {param.numel():,} params")
    else:
        frozen_params += param.numel()

print(f"\n=== Summary ===")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen parameters: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")

In [ ]:


def train_model(model, train_loader, val_loader, eval_iter, eval_freq, num_epochs, optimizer, device):
    
    train_loss, val_loss, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1
    
    for epoch in range(num_epochs):
        model.train() # 1) put the model in training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 2) set the gradients from previous epoch to zero
            loss = crossentropyloss(input_batch, target_batch, model, device) 
            loss.backward() # 3) calculate the gradients
            optimizer.step() # 4) update the model weights using the loss gradients
            examples_seen += input_batch.shape[0] # track the examples instead of tokens
            global_step += 1

            # 5) optinal evaluation step
            if global_step % eval_freq == 0 :
                train_loss1, val_loss1 = evaluate_model(model,device, eval_iter, train_loader, val_loader)
                train_loss.append(train_loss1)
                val_loss.append(val_loss1)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss[-1]:.3f}, Val loss {val_loss[-1]:.3f}")


        train_acc = calc_acc(train_loader, model, device, num_batches=eval_iter)
        val_acc = calc_acc(val_loader, model, device, num_batches=eval_iter)
        print(f"training accuracy: {train_acc*100: .2f}")
        print(f"validation accuracy: {val_acc*100: .2f}")
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
    return train_loss, val_loss, train_accs, val_accs, examples_seen
    
def evaluate_model(model, device, eval_iter, train_loader, val_loader):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss(train_loader,  model, device, num_batches=eval_iter)
        val_loss = calc_loss(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss



In [ ]:
import time
gpt.to(device)
start_time = time.time()

optimizer = torch.optim.AdamW(gpt.parameters(), lr=4e-4, weight_decay=0.1)

train_loss, val_loss, tokens_seen = train_model(
    gpt, train_loader, val_loader, eval_iter=5, eval_freq=50, num_epochs=1, optimizer=optimizer, device=device)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
torch.save({
    'model_state_dict': gpt.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'checkpoint.pth')

In [ ]:
# Load checkpoint and continue
optimizer = torch.optim.AdamW(gpt.parameters(), lr=4e-4, weight_decay=0.1)
checkpoint = torch.load('checkpoint.pth', map_location=device)
gpt.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

gpt.to(device)

In [ ]:
# Set model to evaluation mode
gpt.eval()

def predict_on_test_set(model, test_loader, device):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for input_batch in test_loader:
            input_batch = input_batch.to(device)
            logits = model(input_batch)[:, -1, :]  # [batch_size, 3]
            
            # Use softmax for proper probability distribution
            probs = torch.softmax(logits, dim=-1)  # [batch_size, 3]
            
            all_predictions.append(probs.cpu().numpy())
    
    predictions = np.concatenate(all_predictions, axis=0)
    return predictions

# Generate predictions on test set
test_predictions = predict_on_test_set(gpt, test_loader, device)

# Create submission DataFrame
if 'id' in df_test.columns:
    test_ids = df_test['id'].values
else:
    test_ids = range(len(test_predictions))

submission_df = pd.DataFrame({
    'id': test_ids,
    'winner_model_a': test_predictions[:, 0],
    'winner_model_b': test_predictions[:, 1], 
    'winner_tie': test_predictions[:, 2]
})

# Save submission file
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head())